# Week 6 Assignment — The Tiny Dungeon

_Build a navigable world from Python data structures_

# One small world, built from everything you know

This week you'll build a tiny text-based dungeon crawler: a program that describes where the player is, shows the exits, takes a command, and updates the world when they move or pick something up. People were writing games like this in the 1970s, and it's still a great first project because the whole thing fits in your head.

The setting doesn't have to be a medieval dungeon. It could be a haunted library or a museum after closing, or any other place you can describe in text.

This is a **capstone**, not a week of new syntax. Everything you need, you already have:

- variables, strings, input, output, and f-strings;
- functions, conditions, and a `while` loop;
- lists and dictionaries; and
- planning, debugging, explaining, and revising a program.

So keep it small enough that you understand every line. Why? Because a three-room adventure that works from start to finish shows everything this assignment is looking for, and a thirty-room epic that never quite runs doesn't.

# What counts as complete

Your adventure needs:

1. At least **three connected locations**.
2. A dictionary that stores the layout and information about each location.
3. A `while` loop that keeps the adventure running.
4. Commands that let the player look, move, view inventory, and quit.
5. At least one object the player can collect.
6. At least one obstacle or ending that checks the player's location or inventory.
7. At least three functions with clear, different jobs.
8. A creative text interface that uses a triple-quoted string (`"""..."""`) for a title screen, room display, ending, or other multi-line output.
9. Evidence that you tested the important paths and revised one thing after watching someone play.

Each of these counts on its own. If the whole adventure isn't finished, submit the pieces that work and say what you'd do next.

# Keep it tiny

All of these are optional. None of them are needed for full credit, and I'd gently steer you away from them until the basic path works:

- combat, enemies, health, or random events;
- more than three or four locations;
- graphics, sound, or elaborate ASCII art;
- commands written as full sentences;
- character classes or object-oriented programming;
- saving to a file, importing libraries, or using outside data.

Build one short path from the entrance to an ending. Add something extra only after that path works.

# Checkpoint 1 — Sketch the world

Plan a world small enough to finish. Every location needs at least one connection to another one, or the player gets stuck.

Planners will probably fill this in first. If you're more of a bricoleur, it's fine to poke at the starter world below and come back to the table once you know what you're making. Either way, you'll submit it.

| Location key | What the player sees | Exits | Object or obstacle |
|---|---|---|---|
| `entry` | | | |
| | | | |
| | | | |

Write the shortest path from the entrance to the ending. Include where the player must collect an object, if your ending requires one.

> Write the shortest path here.

# Checkpoint 2 — Put the layout in a dictionary

A dictionary can connect a short location key like `"entry"` with everything the program needs to know about that place. Each location below has a title, a description, exits, and an item.

Read this starter world from the outside in:

- `ROOMS` is one dictionary representing the whole world.
- `ROOMS["entry"]` is a dictionary representing one location.
- `ROOMS["entry"]["exits"]` is another dictionary connecting directions to location keys.

Change the names and descriptions, then add or revise locations to make the world your own.

In [ ]:
ROOMS = {
    "entry": {
        "title": "THE ENTRY HALL",
        "description": """Rain taps on the roof.
A sealed door stands to the north. A narrow passage leads east.""",
        "exits": {
            "east": "workshop",
            "north": "archive",
        },
        "item": None,
    },
    "workshop": {
        "title": "THE ABANDONED WORKSHOP",
        "description": """Unfinished machines cover the benches.
Something brass glints underneath a folded blueprint.""",
        "exits": {"west": "entry"},
        "item": "brass key",
    },
    "archive": {
        "title": "THE IMPOSSIBLE ARCHIVE",
        "description": """The sealed door swings open.
The book you came to find is waiting on a reading stand.""",
        "exits": {"south": "entry"},
        "item": None,
        "locked_by": "brass key",
        "ending": True,
    },
}

The starter has exactly three locations. If you pick up the key while experimenting and want a fresh start, rerun the `ROOMS` cell and the world resets.

## Explore before building

Predict each result, then run the cell.

In [ ]:
print(ROOMS["entry"]["title"])
print(ROOMS["entry"]["description"])
print(ROOMS["entry"]["exits"])
print(ROOMS["entry"]["exits"]["east"])

Add one `print` of your own that retrieves a single fact from a nested dictionary:

In [ ]:
# YOUR PRINT STATEMENT HERE

# Checkpoint 3 — Make a multi-line interface

Three quotation marks let a string run across several lines, which is exactly what you want for title screens, instructions, room descriptions, menus, dialogue, and endings.

Put an `f` before the opening quotes and a triple-quoted string becomes an f-string, so values in braces get dropped into the bigger display.

In [ ]:
def show_room(current_room, rooms):
    room = rooms[current_room]
    exit_words = ", ".join(room["exits"].keys())

    if room["item"] is None:
        item_words = "nothing to collect"
    else:
        item_words = room["item"]

    screen = f"""
+----------------------------------------+
| LOCATION: {room["title"]}
+----------------------------------------+
{room["description"]}

EXITS: {exit_words}
ITEM:  {item_words}
+----------------------------------------+
"""
    print(screen)


show_room("entry", ROOMS)

Why bother with labels like `EXITS:` and `ITEM:`? Because the border is just decoration. It wraps oddly on a narrow screen, and a screen reader can read it out as a long string of plus signs and dashes. The labels are what tell the player what's there. So make your interface as expressive as you like, but don't let spacing, pictures, or color be the only way to find out about an exit or an object.

Create a triple-quoted title screen for your adventure. Include the title, the player's goal, and the available command words.

In [ ]:
INTRO = """
==========================================
           YOUR TITLE GOES HERE
==========================================
Goal: Write one clear sentence here.

Commands: look, north, south, east, west,
          take, inventory, help, quit
==========================================
"""

print(INTRO)

# Checkpoint 4 — Turn commands into actions

The parts of the game that change as you play can live in ordinary variables and a list:

In [ ]:
current_room = "entry"
inventory = []
playing = True

## Move through the layout

`move_player` looks up the current location's exits. It returns the next location key and a message, and if the move isn't possible it returns the current key unchanged so the player stays put.

In [ ]:
def move_player(current_room, direction, rooms, inventory):
    room = rooms[current_room]

    if direction not in room["exits"]:
        return current_room, "There is no exit in that direction."

    next_room = room["exits"][direction]
    needed_item = rooms[next_room].get("locked_by")

    if needed_item is not None and needed_item not in inventory:
        return current_room, f"That way is locked. You need the {needed_item}."

    return next_room, f"You move {direction}."

Try one movement that works and one that doesn't:

In [ ]:
test_inventory = []

new_room, message = move_player("entry", "east", ROOMS, test_inventory)
print(new_room, message)

new_room, message = move_player("entry", "west", ROOMS, test_inventory)
print(new_room, message)

## Collect an object

Finish `take_item`. When the player picks something up, add it to `inventory` and set the room's item to `None`. That second part is what stops the player from taking the same brass key forty times.

In [ ]:
def take_item(current_room, rooms, inventory):
    room = rooms[current_room]
    item = room["item"]

    if item is None:
        return "There is nothing here to take."

    # Add the item to inventory.
    # Change the room's item to None.
    # Return a message naming the collected item.

This function changes two data structures at once: the inventory list and one dictionary inside `ROOMS`. Sketch what each one holds before and after:

| Moment | `inventory` | `ROOMS[current_room]["item"]` |
|---|---|---|
| Before `take_item` | | |
| After `take_item` | | |

# Checkpoint 5 — Assemble the adventure loop

The loop below is mostly a traffic director. The real work happens in functions you've already written, so each branch only has to call the right one.

Complete `take_item`, personalize the world, and then run this cell.

This cell starts the player over, but it doesn't reset the world. If you've already picked up the key in an earlier run, **rerun the `ROOMS` cell first**, or the key will be gone and the game can't be won.

In [ ]:
current_room = "entry"
inventory = []
playing = True

print(INTRO)

while playing:
    show_room(current_room, ROOMS)
    command = input("What do you do? ").strip().lower()

    if command == "look":
        print("You take another careful look around.")
    elif command in ["north", "south", "east", "west"]:
        current_room, message = move_player(
            current_room, command, ROOMS, inventory
        )
        print(message)
    elif command == "take":
        print(take_item(current_room, ROOMS, inventory))
    elif command == "inventory":
        if len(inventory) == 0:
            print("INVENTORY: empty")
        else:
            print("INVENTORY:", ", ".join(inventory))
    elif command == "help":
        print("Commands: look, north, south, east, west, take, inventory, help, quit")
    elif command == "quit":
        print("You leave the adventure for another day.")
        playing = False
    else:
        print("I do not understand that command. Type help for the command list.")

    if ROOMS[current_room].get("ending", False):
        show_room(current_room, ROOMS)   # let the player see where they ended up
        print("""
==========================================
                 THE END
==========================================
Write your ending here.
==========================================
""")
        playing = False

Do you see what's *not* in the loop? There's no separate branch for each location. So how does the game know you're in the workshop and not the entry hall? Changing `current_room` changes which dictionary the same functions read. That's why you can add a fourth room by adding one more entry to `ROOMS` (and an exit that leads to it) without touching the loop at all.

# Checkpoint 6 — Test, watch, and revise

Play each of these paths and write down what happened. Fix one at a time.

| Test path | Expected behavior | Observed result | Pass or revision |
|---|---|---|---|
| Type an unknown command | explanation and help reminder | | |
| Try a direction with no exit | message; location stays the same | | |
| Reach the locked location without the object | blocked; location stays the same | | |
| Collect the object twice | second attempt finds nothing | | |
| View an empty and non-empty inventory | both are clearly labeled | | |
| Take the shortest successful path | ending appears and loop stops | | |

Before each playtest (and each fresh test path), rerun the `ROOMS` cell and then this loop cell, so every player starts with a fresh world.

Ask another person to play, and resist the urge to explain the controls. Wherever they get stuck is exactly what you want to find out about.

1. Where did they become confused about a command, exit, object, or goal?
2. What one concrete part of the interface or world did you revise?
3. Can the whole adventure be played with a keyboard?
4. Does every important cue make sense without relying on color, pictures, or spacing?

> Write your playtest notes here.

# Explain the program you built

Answer briefly:

1. Why is a dictionary a useful representation of this world?
2. What value changes when the player moves? What structures change when the player takes an object?
3. Choose one function and explain its inputs, its returned value, and any list or dictionary it changes.
4. What would you add if you had one more hour? What would you deliberately leave out?

# Responsible reuse

Tell me about any substantial copied or AI-generated code: where it came from, what you changed, and how you tested it. If there isn't any, say so. This isn't a trap; it's how I know what you understand.

> Write your responsible-reuse note here.

# What to submit

Submit:

1. Your completed notebook, Colab link, or `.py` file.
2. Your world-planning table and shortest successful path.
3. One copied playthrough from the entrance to an ending.
4. Your completed test table and playtest notes.
5. Your explanation answers and responsible-reuse note.

Submit working partial progress even if the ending isn't reachable yet. Label what works, what doesn't, and the next change you'd make.